# Devoir XAI 2025-2026: Explainable AI
## **Auteur:** ELHADJI Oussama
## **Encadré par:** Mme. S. MHAMMEDI
**Date:** Janvier 2026

Ce notebook contient les solutions complètes pour les 4 exercices du devoir XAI.

In [2]:
import gower
import numpy as np
from sklearn import ensemble
from sklearn.datasets import load_iris

def generate_whatif(x_interest, model, dataset):
    y_interest = model.predict(x_interest)[0]
    y_pred = model.predict(dataset)
    different_pred_mask = (y_pred != y_interest)
    X_different = dataset[different_pred_mask]
    
    if len(X_different) == 0:
        return None
    
    distances = gower.gower_matrix(x_interest, X_different)[0]
    closest_idx = np.argmin(distances)
    counterfactual = X_different[closest_idx].reshape(1, -1)
    
    return counterfactual

def evaluate_counterfactual(counterfactual, x_interest, model):
    y_interest = model.predict(x_interest)[0]
    num_features = x_interest.shape[1]
    non_minimal_features = []
    
    for i in range(num_features):
        cf_modified = counterfactual.copy()
        cf_modified[0, i] = x_interest[0, i]
        y_modified = model.predict(cf_modified)[0]
        
        if y_modified != y_interest:
            non_minimal_features.append(i)
    
    return non_minimal_features

if __name__ == "__main__":
    print("="*80)
    print("EXERCISE 4: What-If Counterfactuals")
    print("="*80)
    
    # Load iris dataset and create binary task
    iris = load_iris()
    X = iris.data
    y = iris.target
    y[y == 2] = 1  # Binary: class 0 vs class 1
    feature_names = iris.feature_names
    
    print("\n--- DATASET INFO ---")
    print(f"Features: {feature_names}")
    print(f"Classes: {np.unique(y, return_counts=True)}")
    
    # Reserve first row for x_interest
    x_interest = X[0, :].reshape(1, -1)
    X = np.delete(X, 0, axis=0)
    y = np.delete(y, 0, axis=0)
    
    # Fit Random Forest
    print("\n--- MODEL TRAINING ---")
    model = ensemble.RandomForestClassifier(random_state=0)
    model.fit(X, y)
    print(f"Model accuracy: {model.score(X, y):.3f}")
    
    # Analyze x_interest
    print("\n--- ORIGINAL INSTANCE ---")
    print(f"Values: {x_interest[0]}")
    pred_interest = model.predict(x_interest)[0]
    proba_interest = model.predict_proba(x_interest)[0]
    print(f"Prediction: {pred_interest}")
    print(f"Probabilities: {proba_interest}")
    
    # Generate counterfactual
    print("\n--- COUNTERFACTUAL ---")
    cf = generate_whatif(x_interest=x_interest, model=model, dataset=X)
    
    if cf is not None:
        print(f"Values: {cf[0]}")
        pred_cf = model.predict(cf)[0]
        proba_cf = model.predict_proba(cf)[0]
        print(f"Prediction: {pred_cf}")
        print(f"Probabilities: {proba_cf}")
        
        # Feature changes
        print("\n--- FEATURE CHANGES ---")
        changes = cf[0] - x_interest[0]
        
        for i, name in enumerate(feature_names):
            orig = x_interest[0, i]
            new = cf[0, i]
            change = changes[i]
            pct = (change / orig * 100) if orig != 0 else 0
            print(f"{name:25s}: {orig:.3f} -> {new:.3f} (Δ={change:+.3f}, {pct:+.1f}%)")
        
        gower_dist = gower.gower_matrix(x_interest, cf)[0, 0]
        print(f"\nGower distance: {gower_dist:.4f}")
        
        # Minimality
        print("\n--- MINIMALITY ---")
        non_minimal = evaluate_counterfactual(cf, x_interest, model)
        
        if len(non_minimal) == 0:
            print("MINIMAL: All changes necessary")
        else:
            print(f"NON-MINIMAL")
            print(f"Unnecessary features: {[feature_names[i] for i in non_minimal]}")
            necessary = [i for i in range(len(feature_names)) if i not in non_minimal]
            print(f"Necessary features: {[feature_names[i] for i in necessary]}")
    
    print("\n" + "="*80)


EXERCISE 4: What-If Counterfactuals

--- DATASET INFO ---
Features: ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
Classes: (array([0, 1]), array([ 50, 100]))

--- MODEL TRAINING ---
Model accuracy: 1.000

--- ORIGINAL INSTANCE ---
Values: [5.1 3.5 1.4 0.2]
Prediction: 0
Probabilities: [1. 0.]

--- COUNTERFACTUAL ---
Values: [5.1 2.5 3.  1.1]
Prediction: 1
Probabilities: [0. 1.]

--- FEATURE CHANGES ---
sepal length (cm)        : 5.100 -> 5.100 (Δ=+0.000, +0.0%)
sepal width (cm)         : 3.500 -> 2.500 (Δ=-1.000, -28.6%)
petal length (cm)        : 1.400 -> 3.000 (Δ=+1.600, +114.3%)
petal width (cm)         : 0.200 -> 1.100 (Δ=+0.900, +450.0%)

Gower distance: 0.7500

--- MINIMALITY ---
NON-MINIMAL
Unnecessary features: ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
Necessary features: []



## **Part b: Attributes and Advantages/Disadvantages**

**Attributes Satisfied:**

**Validity**: Fully satisfied. The counterfactual is a real data point from the training set that the model actually predicts as Class 1, guaranteeing the prediction is achievable.

**Proximity**: Satisfied. The method uses Gower distance (0.7500) to find the closest instance with different prediction, minimizing the changes needed.

**Sparsity**: Partially satisfied. Three out of four features changed (sepal width, petal length, petal width), with sepal length remaining constant.

**Attributes NOT Satisfied:**

**Parsimony/Minimality**: Failed. The evaluation shows all features are marked as unnecessary, meaning this counterfactual requires more changes than the minimum needed to flip the prediction.

**Actionability**: Not addressed. The method doesn't consider whether changes are realistic (e.g., increasing petal width by 450% may not be biologically feasible).

**Diversity**: Not addressed. Only one counterfactual is generated, providing no alternative paths.

**Advantages:**
- Simple implementation using nearest neighbor search
- Always valid since it uses real data points
- Fast computation with Gower distance handling mixed data types
- No optimization needed

**Disadvantages:**
- Non-minimal explanations (unnecessary feature changes included)
- Limited to existing data points (cannot interpolate)
- No guarantee of actionable changes
- Single counterfactual lacks diversity
- May be far from decision boundary if training data is sparse

***

## **Question 1: Feature modifications**

The counterfactual modifies three features while keeping sepal length constant. Sepal width decreases by 1.0 cm (-28.6%), moving from 3.5 to 2.5 cm. Petal length increases by 1.6 cm (+114.3%), from 1.4 to 3.0 cm. Petal width shows the most dramatic change, increasing by 0.9 cm (+450%), from 0.2 to 1.1 cm. The original instance has small petals characteristic of Class 0 (setosa), while the counterfactual has larger petals typical of Class 1 (versicolor/virginica).

***

## **Question 2: Most changed features**

Petal width changed most in relative terms (+450%), followed by petal length (+114.3%). In absolute terms, petal length changed by 1.6 cm versus 0.9 cm for petal width. Sepal width decreased moderately (-28.6%), while sepal length remained unchanged. The petal dimensions drive the class change, suggesting these are the critical features for this model's decision boundary near the original instance.

***

## **Question 3: Model sensitivity**

The model is highly sensitive to petal dimensions and insensitive to sepal length. The fact that all features are marked as unnecessary in the minimality test reveals complex feature interactions - no single feature change alone is sufficient to flip the prediction. The model likely uses a combination rule requiring simultaneous changes in multiple features. The large percentage changes needed (450% for petal width) suggest the original instance sits far from the decision boundary, requiring substantial movement to reach the opposite class. The perfect training accuracy (100%) indicates potential overfitting, which may explain why the nearest opposite-class point is relatively distant (Gower distance 0.75).
